In [0]:
#Buscando as informações necessárias do .env
import os
from dotenv import load_dotenv

load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

assert all([client_id, tenant_id, client_secret]), "Faltou preencher alguma variavel no .env"

In [0]:
#Verificação do container e do caminho
import adlfs

storage_options = {
    "account_name": storage_account_name,
    "client_id": client_id,
    "client_secret": client_secret,
    "tenant_id": tenant_id,
}

fs = adlfs.AzureBlobFileSystem(**storage_options)

real_time_ref_path = f"{container_name}/real-time-data"

print(f"Container: {container_name}")
print(f"Caminho de referencia (real-time): {real_time_ref_path}")

In [0]:
#Polling para verificar a atualização dos dados raw em real time, são 5 tentativas a cada 30 segundos. 
import time


def check_real_time_data(max_attempts=5, wait_seconds=30):
    for attempt in range(1, max_attempts + 1):
        arquivos = fs.find(real_time_ref_path)
        if arquivos:
            print(f"Dados encontrados na tentativa {attempt}:")
            return arquivos
        print(f"Tentativa {attempt}/{max_attempts}: nada em {real_time_ref_path} ainda.")
        if attempt < max_attempts:
            time.sleep(wait_seconds)
    print("Nada encontrado ainda. Rode esta celula de novo mais tarde.")
    return []


arquivos_encontrados = check_real_time_data()

In [0]:
#Conexão no SQL Server

sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")

assert all([sql_host, sql_database, sql_username, sql_password]), "Faltou preencher alguma variavel de SQL Server no .env"

jdbc_url = (
    f"jdbc:sqlserver://{sql_host}:1433;"
    f"database={sql_database};encrypt=true;trustServerCertificate=false;loginTimeout=30;"
)

jdbc_properties = {
    "user": sql_username,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}